# 🔍 Pipeline de Análisis de Incidentes de IA

Este notebook analiza un dataset estructurado y genera:
- Un **reporte HTML interactivo** para explorar los datos (filtros, gráficos, wordclouds)
- El **dataset procesado** en formato Parquet
- Un **archivo de métricas** en JSON

---

## ¿Cómo usar este notebook?

1. **Celda 1 — Setup**: instalá las dependencias (solo la primera vez). Colab se reinicia automáticamente, es normal.
2. **Celda 2 — Configuración**: editá con tu CSV y tus preferencias. Es la única celda que necesitás tocar.
3. **Celdas 3 en adelante**: ejecutalas en orden sin modificar nada.

> 💡 **Tip de velocidad**: Si usás análisis de sentimiento con modelo transformer, activá GPU en
> `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`. Reduce el tiempo de ~10 min a ~2 min.

## Celda 1 — Setup

⚠️ Ejecutá esta celda **una sola vez**. Al terminar, Colab se reinicia automáticamente.
Cuando el entorno se reinicie, **no vuelvas a correr esta celda** — pasá directo a la Celda 2.

In [ ]:
import os

REPO_URL = "https://github.com/karenrg/incidents_pipeline"
REPO_DIR = "incidents_pipeline"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"'{REPO_DIR}' ya existe, saltando clone.")

!pip install -r {REPO_DIR}/requirements_colab.txt -q

print("\n✅ Instalación completa.")
print("Colab se va a reiniciar ahora. Cuando veas el mensaje de reinicio, continuá desde la Celda 2.")
os.kill(os.getpid(), 9)

## Celda 2 — Configuración

**Esta es la única celda que necesitás editar.**

- Lo marcado con `# ← REQUERIDO` es obligatorio.
- Todo lo demás es opcional y ya tiene valores por defecto.

In [ ]:
import os, sys, yaml
from pathlib import Path

# ── Posicionarse en el repositorio ───────────────────────────────────────────
REPO_DIR = "incidents_pipeline"
if not os.path.exists(REPO_DIR):
    raise RuntimeError("No se encontró el repositorio. Volvé a correr la Celda 1.")
os.chdir(REPO_DIR)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", os.getcwd())


# ════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — Editá esta sección
# ════════════════════════════════════════════════════════════════════════════

# ── 1. ARCHIVO DE ENTRADA ────────────────────────────────────────────────────
# Ruta al CSV con los datos a analizar. Para usar un archivo de Drive:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   SOURCE_PATH = "/content/drive/MyDrive/mi_dataset.csv"

SOURCE_PATH = "data/raw/aiid_incidents.csv"  # ← REQUERIDO


# ── 2. MAPEO DE COLUMNAS ─────────────────────────────────────────────────────
# Decile al pipeline qué columna de tu CSV cumple cada rol.
# Ponés el nombre exacto de la columna en tu CSV (el de la derecha).
# Si tu dataset no tiene alguna columna, dejala como "" (cadena vacía).

COLUMN_MAPPING = {
    "text_data":  "description",           # ← REQUERIDO: texto principal
    "event_date": "date",                  # ← REQUERIDO: fecha
    "geo_zone":   "Location Region",       # Región geográfica (opcional)
    "tags_list":  "Risk Domain",           # Categorías o etiquetas (opcional)
    "harmtype":   "Risk Subdomain",        # Subtipo o categoría secundaria (opcional)
    "industries": "Sector of Deployment",  # Industria o sector (opcional)
    "harmed":     "harmed",                # Grupos afectados (opcional)
    "harmlevel":  "AI Harm Level",         # Nivel de gravedad (opcional)
}

# Columnas que contienen listas de valores (ej: '["sector1", "sector2"]')
# Dejá vacío si no aplica: MULTILABEL_COLUMNS = []
MULTILABEL_COLUMNS = ["industries"]


# ── 3. FILTROS (opcional) ────────────────────────────────────────────────────
# Rango de años a analizar. None = sin límite.
YEAR_RANGE = [2014, 2023]

# Regiones a incluir. Lista vacía = todas.
REGIONS = ["North America", "Europe", "Asia"]


# ── 4. ANÁLISIS DE SENTIMIENTO (opcional) ────────────────────────────────────
# Clasifica el texto de cada registro como negativo alto/medio/bajo.
# Requiere tiempo extra. Dejalo en False si no lo necesitás.

SENTIMENT_ENABLED = False

# Si SENTIMENT_ENABLED = True, elegí el modelo:
#   "openai"      → GPT-4o-mini (necesita API key, más preciso)
#   "transformer" → Modelo HuggingFace (sin API key, ~500 MB de descarga)
#                   El modelo se configura con model_name en params.yaml
#                   Default: cardiffnlp/twitter-roberta-base-sentiment-latest
SENTIMENT_BACKEND = "openai"

# Solo si SENTIMENT_BACKEND = "openai"
# Recomendado: guardá tu key en Secrets de Colab (ícono 🔑 en el panel izquierdo)
# y descomentá la línea de abajo en lugar de pegarla directamente aquí.
# from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = ""   # ← pegá tu key aquí solo si no usás Secrets


# ── 5. REPORTE (opcional) ────────────────────────────────────────────────────
REPORT_TITLE      = "Análisis de Incidentes de IA"
DATA_SOURCE_LABEL = "AI Incident Database (AIID)"


# ════════════════════════════════════════════════════════════════════════════
# Aplicar configuración al pipeline — no modificar esta sección
# ════════════════════════════════════════════════════════════════════════════

with open("config/params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["data"]["source_path"]            = SOURCE_PATH
config["data"]["column_mapping"]         = COLUMN_MAPPING
config["data"]["multilabel_columns"]     = MULTILABEL_COLUMNS
config["filters"]["year_range"]          = YEAR_RANGE
config["filters"]["regions"]             = REGIONS
config["sentiment"]["enabled"]           = SENTIMENT_ENABLED
config["sentiment"]["backend"]           = SENTIMENT_BACKEND
config["reporting"]["report_title"]      = REPORT_TITLE
config["reporting"]["data_source_label"] = DATA_SOURCE_LABEL

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
elif SENTIMENT_ENABLED and SENTIMENT_BACKEND == "openai":
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
    except Exception:
        print("⚠️  No se encontró OPENAI_API_KEY. Configurala en Secrets de Colab o pegala en OPENAI_API_KEY.")

print("✅ Configuración aplicada:")
print(f"   Dataset:     {SOURCE_PATH}")
print(f"   Años:        {YEAR_RANGE}")
print(f"   Regiones:    {REGIONS}")
print(f"   Sentimiento: {'activado (' + SENTIMENT_BACKEND + ')' if SENTIMENT_ENABLED else 'desactivado'}")
print(f"   Reporte:     {REPORT_TITLE}")

## Celda 3 — Verificar entorno

In [ ]:
import torch

device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"Dispositivo disponible: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif config["sentiment"]["enabled"] and config["sentiment"]["backend"] == "transformer":
    print("Sin GPU. El paso de sentimiento tardará ~10 min en CPU.")

## Celda 4 — Imports e inicialización

In [ ]:
from pathlib import Path

from src import configure_logging, set_global_seeds
from src.ingestion import load_and_validate
from src.preprocessing import preprocess
from src.nlp import process_text
from src.sentiment import run_sentiment
from src.analysis import run_analysis
from src.html_report import generate_html_report

configure_logging()
set_global_seeds(config["random_state"])
print("✅ Listo.")

## Celda 5 — Ingestión

Carga el CSV y renombra las columnas según el mapeo configurado.

In [ ]:
df = load_and_validate(config)
print(f"Dataset cargado: {len(df):,} filas, {len(df.columns)} columnas")
df.head()

## Celda 6 — Preprocesamiento

Limpieza, parseo de fechas, filtros por año y región.

In [ ]:
df = preprocess(df, config)
print(f"Después del preprocesamiento: {len(df):,} filas")
df.head()

## Celda 7 — NLP

Tokenización, lematización y detección de keywords temáticas.

In [ ]:
df = process_text(df, config)
df[["tokens", "mental_health_flag"]].head()

## Celda 8 — Análisis de sentimiento

Se omite automáticamente si `SENTIMENT_ENABLED = False`.

> Con `backend=transformer`: descarga el modelo (~500 MB) la primera vez. Con GPU ~2 min, sin GPU ~10 min.

In [ ]:
if config["sentiment"]["enabled"]:
    df = run_sentiment(df, config)
    print("Sentimiento calculado.")
    display(df[["sentiment_score", "sentiment_label"]].head())
else:
    print("Sentimiento desactivado. Para activarlo, ponés SENTIMENT_ENABLED = True en la Celda 2.")

## Celda 9 — Guardar dataset procesado

In [ ]:
processed_path = Path(config["data"]["processed_path"])
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path)
print(f"✅ Dataset guardado en: {processed_path}")

## Celda 10 — Análisis descriptivo

Calcula todas las métricas: distribuciones, rankings, tendencias temporales, etc.

In [ ]:
metrics = run_analysis(df, config)
print("✅ Métricas calculadas.")

## Celda 11 — Reporte HTML interactivo

Genera un archivo HTML con explorador de datos, filtros dinámicos, gráficos interactivos y wordclouds.

In [ ]:
html_path = generate_html_report(df, metrics, config)
print(f"✅ Reporte HTML generado en: {html_path}")

## Celda 12 — Descargar resultados

Descarga los archivos generados a tu computadora.

In [ ]:
from google.colab import files

print("Descargando archivos...")
files.download(str(html_path))
files.download("outputs/reports/metrics.json")

print("\n✅ Pipeline completado. Archivos descargados:")
print(f"   🌐 {html_path.name}")
print(f"   📊 metrics.json")